# CSV to Iceberg Migration Example

This notebook demonstrates how to migrate CSV files to Apache Iceberg tables, a common use case when transitioning from traditional data formats to modern table formats.

## Overview

Migrating CSV data to Iceberg provides several benefits:
- **Better performance**: Columnar Parquet format vs row-based CSV
- **Schema evolution**: Add/modify columns without breaking existing queries
- **ACID transactions**: Reliable data operations with rollback support
- **Time travel**: Query historical data at any point in time
- **Partitioning**: Efficient data organization for large datasets

## Migration Strategies

1. **Simple migration**: Direct CSV to Iceberg conversion
2. **Schema evolution**: Enhance schema during migration
3. **Partitioned migration**: Organize data by partition keys
4. **Incremental migration**: Handle ongoing CSV updates

In [ ]:
# Import required libraries
import os
import tempfile
import csv
import pyarrow as pa
import pyarrow.csv as csv_pa

import pyiceberg
from pyiceberg.catalog import load_catalog

print(f"PyIceberg version: {pyiceberg.__version__}")
print(f"PyArrow version: {pa.__version__}")

## Step 1: Create Sample CSV Data

First, let's create sample CSV files that simulate real-world data that needs to be migrated.

In [ ]:
# Create a temporary directory for CSV files
csv_dir = tempfile.mkdtemp(prefix="csv_data_")
print(f"CSV directory: {csv_dir}")

# Create sample sales CSV data
sales_csv_path = os.path.join(csv_dir, "sales.csv")

sales_data = [
    ["transaction_id", "customer_id", "product_id", "quantity", "unit_price", "transaction_date"],
    ["1", "101", "501", "2", "10.00", "2024-01-01"],
    ["2", "102", "502", "1", "25.00", "2024-01-02"],
    ["3", "101", "501", "3", "10.00", "2024-01-01"],
    ["4", "103", "503", "1", "50.00", "2024-01-03"],
    ["5", "102", "502", "2", "25.00", "2024-01-02"],
    ["6", "104", "504", "1", "100.00", "2024-01-04"],
    ["7", "101", "501", "4", "10.00", "2024-01-05"],
    ["8", "105", "505", "2", "75.00", "2024-01-03"],
    ["9", "103", "503", "1", "50.00", "2024-01-04"],
    ["10", "102", "502", "3", "25.00", "2024-01-05"],
]

with open(sales_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(sales_data)

print(f"Created sample CSV file: {sales_csv_path}")

# Display the CSV content
print("\nCSV content:")
with open(sales_csv_path, 'r') as f:
    print(f.read())

## Step 2: Setup Iceberg Catalog

Create an Iceberg catalog to store the migrated table.

In [ ]:
# Create a temporary warehouse location
warehouse_path = tempfile.mkdtemp(prefix="iceberg_warehouse_")
print(f"Warehouse location: {warehouse_path}")

# Configure and load the catalog
catalog = load_catalog(
    "default",
    type="sql",
    uri=f"sqlite:///{warehouse_path}/pyiceberg_catalog.db",
    warehouse=f"file://{warehouse_path}",
)

print("Catalog loaded successfully!")

# Create a namespace
catalog.create_namespace("default")
print(f"Available namespaces: {list(catalog.list_namespaces())}")

## Step 3: Read CSV with PyArrow

Use PyArrow to read the CSV file and convert it to a format suitable for Iceberg.

In [ ]:
# Read CSV using PyArrow
csv_table = csv_pa.read_csv(sales_csv_path)

print("CSV data loaded with PyArrow:")
print(csv_table)
print(f"\nSchema: {csv_table.schema}")
print(f"Total rows: {len(csv_table)}")

# Convert types if needed (PyArrow infers types, but we can be explicit)
# For example, ensure transaction_id and customer_id are integers
csv_table = csv_table.cast(pa.schema([
    pa.field("transaction_id", pa.int64()),
    pa.field("customer_id", pa.int64()),
    pa.field("product_id", pa.int64()),
    pa.field("quantity", pa.int64()),
    pa.field("unit_price", pa.float64()),
    pa.field("transaction_date", pa.string())
]))

print("\nConverted schema:")
print(csv_table.schema)

## Step 4: Create Iceberg Table and Migrate Data

Create the Iceberg table with the CSV schema and migrate the data.

In [ ]:
# Create Iceberg table with the CSV schema
table = catalog.create_table(
    "default.sales",
    schema=csv_table.schema,
)

print(f"Created Iceberg table: {table}")
print(f"Table location: {table.location()}")
print(f"Table schema: {table.schema()}")

# Migrate the data from CSV to Iceberg
table.append(csv_table)
print(f"\nData migration completed!")
print(f"Rows in Iceberg table: {len(table.scan().to_arrow())}")

## Step 5: Verify Migration

Verify that the data was migrated correctly by querying the Iceberg table.

In [ ]:
# Read the migrated data from Iceberg
migrated_data = table.scan().to_arrow()

print("Data from Iceberg table:")
print(migrated_data)
print(f"\nTotal rows: {len(migrated_data)}")
print(f"Schema: {migrated_data.schema}")

# Compare with original CSV data
print("\nOriginal CSV rows:", len(csv_table))
print("Migrated Iceberg rows:", len(migrated_data))
print("Migration successful:", len(csv_table) == len(migrated_data))

## Advanced Migration: Schema Enhancement

One of Iceberg's key benefits is schema evolution. Let's enhance the schema during migration by adding computed columns.

In [ ]:
import pyarrow.compute as pc

# Add computed columns to enhance the schema
enhanced_table = csv_table

# Add total_amount column (quantity * unit_price)
enhanced_table = enhanced_table.append_column(
    "total_amount", 
    pc.multiply(enhanced_table["quantity"], enhanced_table["unit_price"])
)

print("Enhanced schema with computed column:")
print(enhanced_table.schema)
print("\nData with new column:")
print(enhanced_table)

In [ ]:
# Create a new table with the enhanced schema
enhanced_table_iceberg = catalog.create_table(
    "default.sales_enhanced",
    schema=enhanced_table.schema,
)

print(f"Created enhanced table: {enhanced_table_iceberg}")

# Migrate the enhanced data
enhanced_table_iceberg.append(enhanced_table)
print(f"Enhanced data migrated successfully!")
print(f"Rows in enhanced table: {len(enhanced_table_iceberg.scan().to_arrow())}")

## Migration with Partitioning

For larger datasets, partitioning improves query performance. Let's create a partitioned table.

In [ ]:
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform, DayTransform

# Create a partition spec (partition by transaction_date)
partition_spec = PartitionSpec(
    PartitionField(
        source_id=5,  # transaction_date field index
        field_id=1000,
        transform=IdentityTransform(),
        name="transaction_date"
    )
)

# Create a partitioned table
partitioned_table = catalog.create_table(
    "default.sales_partitioned",
    schema=enhanced_table.schema,
    partition_spec=partition_spec
)

print(f"Created partitioned table: {partitioned_table}")
print(f"Partition spec: {partitioned_table.spec()}")
print(f"Partition fields: {list(partitioned_table.spec().fields)}")

# Migrate data to partitioned table
partitioned_table.append(enhanced_table)
print(f"\nData migrated to partitioned table!")
print(f"Rows: {len(partitioned_table.scan().to_arrow())}")

## Best Practices for CSV Migration

### Data Quality Checks
- **Validate CSV structure**: Ensure consistent column names and types
- **Handle missing values**: Decide on null handling strategy
- **Check for duplicates**: Identify and handle duplicate records
- **Validate data ranges**: Ensure values fall within expected ranges

### Schema Design
- **Use appropriate types**: Choose the most efficient data types
- **Add computed columns**: Enhance data with derived values during migration
- **Consider partitioning**: Plan partition strategy for large datasets
- **Document changes**: Keep track of schema evolution

### Performance Considerations
- **Batch size**: Process large CSV files in batches
- **Memory management**: Be mindful of memory for large files
- **File size optimization**: Target appropriate Iceberg file sizes (typically 128MB-1GB)
- **Compression**: Use compression for storage efficiency

### Production Considerations
- **Incremental updates**: Plan for ongoing CSV updates
- **Backward compatibility**: Ensure queries work during migration
- **Monitoring**: Track migration progress and data quality
- **Rollback plan**: Have a strategy to revert if needed

## Conclusion

This example demonstrated three approaches to CSV to Iceberg migration:

1. **Simple Migration**: Direct CSV to Iceberg conversion
2. **Schema Enhancement**: Adding computed columns during migration
3. **Partitioned Migration**: Organizing data for better performance

### Key Benefits of Migrating to Iceberg

- **Performance**: Columnar Parquet format provides better compression and query performance
- **Schema Evolution**: Add/modify columns without breaking existing queries
- **ACID Transactions**: Reliable data operations with rollback support
- **Time Travel**: Query historical data at any point in time
- **Partitioning**: Efficient data organization for large datasets
- **Compatibility**: Works with multiple compute engines (Spark, DuckDB, Trino, etc.)

### Next Steps

- Explore other migration patterns (Parquet, JSON, Avro to Iceberg)
- Implement incremental migration for ongoing CSV updates
- Set up monitoring and data quality checks
- Integrate with your existing data pipeline

## Cleanup

Let's clean up the temporary resources created during this example.

In [ ]:
# Clean up temporary directories
import shutil

try:
    shutil.rmtree(csv_dir)
    print(f"Cleaned up CSV directory: {csv_dir}")
except Exception as e:
    print(f"CSV cleanup warning: {e}")

try:
    shutil.rmtree(warehouse_path)
    print(f"Cleaned up warehouse directory: {warehouse_path}")
except Exception as e:
    print(f"Warehouse cleanup warning: {e}")

print("CSV migration example completed successfully!")